In [1]:
import napari
from tifffile import imread
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import trackpy as tp

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [2]:
# Load image

image = imread('/Users/abamford/Desktop/TUBB2B-KI_preliminary_analysis/Batch20230223/D21/Denoised/Masks_created/100tp_561-100-50ms-1000g_8_conf561_merged.tif')

In [3]:
print(image.dtype, image.shape, np.min(image), np.max(image))

uint16 (100, 2, 1024, 1024) 1 35675


In [4]:
# Loads image mask

image_mask = imread('/Users/abamford/Desktop/TUBB2B-KI_preliminary_analysis/Batch20230223/D21/ROIs/Cytoplasm/ROIs_as_mask_BIOP/100tp_561-100-50ms-1000g_8_conf561_merged_ROIs.tif')

In [5]:
print(image_mask.dtype, image_mask.shape)

float64 (1024, 1024)


In [6]:
# Initializes Napari viewer

viewer = napari.Viewer()

In [7]:
# Adds image to Napari viewer

viewer.add_image(image)

<Image layer 'image' at 0x173acc580>

In [8]:
# Adds mask image to Napari viewer

viewer.add_image(image_mask,
                 colormap="gist_earth",
                 contrast_limits=[0,8],
                 opacity=0.2)

<Image layer 'image_mask' at 0x17b5c0880>

In [9]:
# Loads detected spots

spots_path = r"/Users/abamford/Desktop/TUBB2B-KI_preliminary_analysis/Batch20230223/D21/Spot_detection/Run20231206/1/100tp_561-100-50ms-1000g_8_conf561_merged_spots.csv"
spots = pd.read_csv(spots_path)
spots.head()

,frame,roi_id,x,y,bg_denoised,amp_denoised,sigma_y,sigma_x,mean_spot_intensity
0,0,0.003922,822.425302,283.066468,3332.212670,5855.696239,1.628831,1.547091,925.583333
1,0,0.003922,853.197427,288.643843,4170.516238,6210.705913,1.583591,1.134106,1070.055556
2,0,0.003922,841.557826,290.474610,4601.355514,6377.532870,1.337775,1.333649,1105.611111
3,0,0.003922,834.976349,297.813959,3666.738877,5094.134133,1.459349,1.258551,915.138889
4,0,0.003922,852.897712,317.678090,4737.023620,7732.084374,1.581846,1.428214,1166.722222


In [10]:
# Creates df that contains xyct axes in the same order as image --> tcxy

spots_tcyxcoord = spots.iloc[:, np.r_[0,3,2]]
spots_tcyxcoord.insert(1, "channel", 0)
spots_tcyxcoord

,frame,channel,y,x
0,0,0,283.066468,822.425302
1,0,0,288.643843,853.197427
2,0,0,290.474610,841.557826
3,0,0,297.813959,834.976349
4,0,0,317.678090,852.897712
...,...,...,...,...
4628,99,0,176.169605,875.345442
4629,99,0,180.291319,869.494750
4630,99,0,203.219842,549.619615
4631,99,0,234.689817,612.427273


In [11]:
# Creates array from df (necessary to use in the points layer of Napari)

spots_tcyxarray = spots_tcyxcoord.to_numpy()
spots_tcyxarray

array([[  0.        ,   0.        , 283.06646834, 822.42530168],
       [  0.        ,   0.        , 288.64384278, 853.19742733],
       [  0.        ,   0.        , 290.47460964, 841.55782646],
       ...,
       [ 99.        ,   0.        , 203.21984168, 549.6196152 ],
       [ 99.        ,   0.        , 234.68981665, 612.42727301],
       [ 99.        ,   0.        , 241.18734562, 613.46329474]])

In [12]:
# Adds detected spots to points layer

viewer.add_points(spots_tcyxcoord,
                  face_color='#ffffff00',
                  edge_color='magenta',
                  name='spots_log')

<Points layer 'spots_log' at 0x173f0dca0>

In [13]:
# Loads tracks

tracks_path = r"/Users/abamford/Desktop/TUBB2B-KI_preliminary_analysis/Batch20230223/D21/Tracking/Run20231206/Link5/100tp_561-100-50ms-1000g_8_conf561_merged_spots_tracks.csv"
tracks = pd.read_csv(tracks_path)
tracks.head()

,frame,roi_id,x,y,bg_denoised,amp_denoised,sigma_y,sigma_x,mean_spot_intensity,particle,track_id,unique_id
0,0,0.003922,822.425302,283.066468,3332.212670,5855.696239,1.628831,1.547091,925.583333,0,0.0039215686274509_0.0,122
1,0,0.003922,853.197427,288.643843,4170.516238,6210.705913,1.583591,1.134106,1070.055556,1,0.0039215686274509_1.0,106
2,0,0.003922,841.557826,290.474610,4601.355514,6377.532870,1.337775,1.333649,1105.611111,2,0.0039215686274509_2.0,281
3,0,0.003922,834.976349,297.813959,3666.738877,5094.134133,1.459349,1.258551,915.138889,3,0.0039215686274509_3.0,56
4,0,0.003922,895.177569,382.919539,5425.495784,11274.321011,1.214050,1.788121,1468.916667,4,0.0039215686274509_4.0,110


In [14]:
# Takes columns "particle", "frame", "x", "y" and sorts into format required by napari for visualization
# Inserts channel information right before y and x coordinate, according to image dimensions

tracks = tracks.iloc[:, np.r_[11,0,3,2]].sort_values(["unique_id", "frame"])
tracks.insert(2, "channel", 0)
tracks.head(20)

,unique_id,frame,channel,y,x
3063,0,8,0,453.713335,495.450369
3066,0,9,0,451.714674,496.823248
3069,0,10,0,453.306666,497.046902
3072,0,11,0,453.236459,496.903825
3076,0,12,0,456.859685,497.910793
3080,0,13,0,458.273835,498.610962
3083,0,14,0,457.771619,498.328995
3090,0,18,0,458.315756,499.498626
913,1,88,0,361.526197,879.673108
920,1,89,0,363.538122,876.716449


In [15]:
viewer.add_tracks(tracks[['unique_id', 'frame', 'channel', 'y', 'x']], name='tracks', 
                  properties={
                      'id': tracks['unique_id'].values,
                             },
                  color_by='id',
                  colormap='hsv', 
                 )

/Users/abamford/Gitrepositories/gchao_singlemolecule_tracking/infrastructure/apps/micromamba/envs/tracking_analysis/lib/python3.9/site-packages/napari/layers/tracks/tracks.py:623: UserWarning: Previous color_by key 'id' not present in features. Falling back to track_id
  warn(


<Tracks layer 'tracks' at 0x1746198e0>